# Experiment 7 — High-throughput LLM Serving using vLLM

**Course:** LLM and Agentic AI  ·  **Instructor:** Dr. Kunal Meher

vLLM needs an **NVIDIA GPU (CUDA)**, so this experiment runs on Google Colab's free GPU (not on a Mac).

### ⚠️ BEFORE YOU RUN ANYTHING — enable the GPU
1. Menu: **Runtime → Change runtime type**
2. Hardware accelerator: **T4 GPU** → **Save**
3. Then run the cells **top to bottom** (Shift+Enter on each).

Wherever you see 📸 **SCREENSHOT**, capture that cell together with its output — those become the figures in the report.

Model used: `Qwen/Qwen2.5-0.5B-Instruct` (small + fast, so it loads quickly on the free tier and lets us push high concurrency to see the serving behavior clearly).

## Step 0 — Confirm we actually have a GPU

📸 **SCREENSHOT** this cell's output (proves the experiment ran on a real NVIDIA GPU).

In [ ]:
!nvidia-smi

## Step 1 — Install vLLM and serve a model

First install vLLM (this pulls a large package — give it a few minutes).

In [ ]:
!pip install -q vllm
import vllm, torch
print("vLLM version :", vllm.__version__)
print("Torch / CUDA :", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("GPU          :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

Now launch the vLLM **OpenAI-compatible server** in the background. Logs are written to `vllm_server.log` so we can inspect them in later steps (continuous batching + KV cache).

📸 **SCREENSHOT** this cell's output (shows the server reached `READY` and the startup log tail).

In [ ]:
import subprocess, time, requests

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
LOG   = "vllm_server.log"

_logf = open(LOG, "w")
server = subprocess.Popen(
    ["vllm", "serve", MODEL,
     "--port", "8000",
     "--dtype", "float16",            # T4 has no bf16 → use fp16
     "--gpu-memory-utilization", "0.9",
     "--max-model-len", "4096"],
    stdout=_logf, stderr=subprocess.STDOUT,
)
print("Launched vLLM server — pid:", server.pid)

start, ready = time.time(), False
while time.time() - start < 600:
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            ready = True
            break
    except Exception:
        pass
    time.sleep(3)

print("STATUS:", "READY ✅" if ready else "TIMEOUT ❌", f"(after {time.time()-start:.0f}s)")
print("\n----- last 30 lines of vllm_server.log -----")
print(subprocess.run(["tail", "-n", "30", LOG], capture_output=True, text=True).stdout)

Send one request to confirm the model is really serving.

📸 **SCREENSHOT** this cell's output (a real completion coming back from the served model).

In [ ]:
import requests, json

resp = requests.post(
    "http://localhost:8000/v1/completions",
    json={"model": MODEL,
          "prompt": "In one sentence, what is high-throughput LLM serving?",
          "max_tokens": 60, "temperature": 0.0},
    timeout=120,
).json()

print("Generated text:\n", resp["choices"][0]["text"].strip())
print("\nUsage:", json.dumps(resp["usage"], indent=2))

## Step 2 — Test concurrency

Send increasing numbers of requests **at the same time** and measure latency + throughput. Watch throughput (tokens/s) climb as concurrency rises — that is vLLM serving many requests in parallel instead of one-by-one.

📸 **SCREENSHOT** this cell's output (the concurrency table).

In [ ]:
import requests, time, concurrent.futures as cf

URL, PROMPT, MAX_TOKENS = "http://localhost:8000/v1/completions", "Explain what an operating system does.", 128

def one_request(_):
    t0 = time.time()
    r = requests.post(URL, json={"model": MODEL, "prompt": PROMPT,
                                 "max_tokens": MAX_TOKENS, "temperature": 0.0}, timeout=300)
    return time.time() - t0, r.json().get("usage", {}).get("completion_tokens", 0)

def run_load(n):
    t0 = time.time()
    with cf.ThreadPoolExecutor(max_workers=n) as ex:
        res = list(ex.map(one_request, range(n)))
    wall = time.time() - t0
    lat  = [d for d, _ in res]
    toks = sum(t for _, t in res)
    print(f"concurrency={n:<4} wall={wall:6.2f}s  avg_latency={sum(lat)/len(lat):5.2f}s  "
          f"max_latency={max(lat):5.2f}s  out_tokens={toks:<6} "
          f"throughput={toks/wall:7.1f} tok/s  req/s={n/wall:5.2f}")

print(f"{' vLLM concurrency test ':-^96}")
for n in [1, 5, 10, 20, 50]:
    run_load(n)

## Step 3 — Observe continuous batching

Fire a burst of 60 requests at once, then read the vLLM **scheduler log lines**. vLLM reports `Running: N reqs, Waiting: M reqs` and a live `Avg generation throughput` — this is *continuous batching*: new requests join the running batch each step instead of waiting for a full batch.

📸 **SCREENSHOT** this cell's output (the `Running:` / `Waiting:` / throughput log lines).

In [ ]:
import concurrent.futures as cf, requests, time, subprocess

def hit(_):
    requests.post(URL, json={"model": MODEL, "prompt": PROMPT,
                             "max_tokens": 256, "temperature": 0.7}, timeout=300)

with cf.ThreadPoolExecutor(max_workers=60) as ex:
    list(ex.map(hit, range(60)))
time.sleep(2)

out = subprocess.run(
    ["grep", "-E", "Running:|Waiting:|Avg generation throughput|Avg prompt throughput", LOG],
    capture_output=True, text=True).stdout.strip().splitlines()
print("----- vLLM scheduler / continuous-batching log lines (most recent) -----")
print("\n".join(out[-25:]) if out else "(no scheduler lines yet — re-run this cell)")

## Step 4 — Observe PagedAttention / KV-cache memory

At startup vLLM profiles the GPU and allocates the **KV cache in fixed-size blocks** (PagedAttention). The log reports the KV-cache size, the number of GPU blocks, and the resulting max concurrency. We also read live GPU memory from `nvidia-smi`.

📸 **SCREENSHOT** this cell's output (KV-cache / GPU-blocks lines + nvidia-smi memory).

In [ ]:
import subprocess

print("===== vLLM KV-cache / PagedAttention init lines =====")
out = subprocess.run(
    ["grep", "-iE", "KV cache|GPU blocks|GPU KV|Maximum concurrency|blocks:|memory profiling|gpu_memory_utilization", LOG],
    capture_output=True, text=True).stdout.strip()
print(out if out else "(pattern not found — paste output of:  !cat vllm_server.log | head -60)")

print("\n===== Live GPU memory (nvidia-smi) =====")
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu",
     "--format=csv"], capture_output=True, text=True).stdout)

## Step 5 — Cross-check with vLLM's built-in benchmark tool

vLLM ships a load generator. It reports **request throughput, output-token throughput, TTFT (time to first token), TPOT (time per output token)**, and ITL — the standard serving metrics — independently of our hand-written test above.

📸 **SCREENSHOT** this cell's output (the benchmark summary table).

*(If `vllm bench serve` errors on your vLLM version, run the fallback cell below it.)*

In [ ]:
!vllm bench serve \
  --backend openai \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --host localhost --port 8000 \
  --endpoint /v1/completions \
  --dataset-name random \
  --num-prompts 200 \
  --random-input-len 128 \
  --random-output-len 128 \
  --request-rate inf

**Fallback** (only run this if the cell above failed): download and run the benchmark script directly.

In [ ]:
!wget -q https://raw.githubusercontent.com/vllm-project/vllm/main/benchmarks/benchmark_serving.py
!python benchmark_serving.py \
  --backend openai \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --host localhost --port 8000 \
  --endpoint /v1/completions \
  --dataset-name random \
  --num-prompts 200 \
  --random-input-len 128 \
  --random-output-len 128 \
  --request-rate inf

## Cleanup (optional)

Stops the server and frees the GPU.

In [ ]:
server.terminate()
print("vLLM server stopped.")